# Train Stress Classification From Preprocessed Raw Waveform

Notebook nay duoc cap nhat theo `md/train.md` (V2):
1) Doc du lieu raw sample tu `data/preprocessed`
2) Trich xuat feature HRV theo `source_file`/window
3) Train va so sanh nhieu mo hinh classification

In [ ]:
,
    "RandomForest_balanced": make_pipeline(
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "ExtraTrees_balanced": make_pipeline(
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "HistGradientBoosting": make_pipeline(
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=8,
            max_iter=400,
            random_state=RANDOM_STATE,
        )
    ),

In [ ]:
# 1) Setup & Imports
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
PREPROCESSED_DIR = Path(r"C:\Users\buck\Napplee\StressClassification\data\preprocessed")
FEATURE_DIR = Path(r"C:\Users\buck\Napplee\StressClassification\data\features")
FEATURE_FILE = FEATURE_DIR / "features_hrv.csv"
ARTIFACTS_DIR = Path("artifacts")
REPORTS_DIR = Path("reports")

for p in [FEATURE_DIR, ARTIFACTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Preprocessed directory: {PREPROCESSED_DIR}")
print(f"Feature file output: {FEATURE_FILE}")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")

## 2) Load Du Lieu Tu Preprocessed
Notebook quet de quy `preprocessed/`, validate schema raw sample va gom du lieu de trich xuat feature.

In [ ]:
def discover_data_files(data_dir: Path):
    patterns = ["*.csv", "*.xlsx", "*.xls", "*.json"]
    files = []
    for pattern in patterns:
        files.extend(data_dir.rglob(pattern))
    return sorted([p for p in files if p.is_file()])


def load_single_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported file format: {path}")


if not PREPROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Khong tim thay thu muc preprocessed: {PREPROCESSED_DIR}. Vui long kiem tra lai duong dan."
    )

raw_files = discover_data_files(PREPROCESSED_DIR)
print(f"Tong so file tabular tim thay trong preprocessed: {len(raw_files)}")

if len(raw_files) == 0:
    npy_files = sorted(PREPROCESSED_DIR.rglob("*.npy"))
    msg = [
        f"Khong tim thay file .csv/.xlsx/.json trong {PREPROCESSED_DIR}.",
        f"Tim thay {len(npy_files)} file .npy (neu co).",
        "Ban can export du lieu raw sample co cot Time/Voltage/Peak/source_file vao preprocessed de trich xuat HRV.",
    ]
    raise FileNotFoundError(" ".join(msg))

frames = []
failed_files = []

for fpath in raw_files:
    try:
        df_i = load_single_file(fpath)
        if df_i is not None and not df_i.empty:
            df_i = df_i.copy()
            if "source_file" not in df_i.columns:
                df_i["source_file"] = str(fpath.relative_to(PREPROCESSED_DIR))
            frames.append(df_i)
    except Exception as ex:
        failed_files.append((str(fpath), str(ex)))

if len(frames) == 0:
    raise ValueError(
        "Tat ca file trong preprocessed deu rong hoac loi khi doc. Kiem tra lai dinh dang du lieu."
    )

raw_df = pd.concat(frames, axis=0, ignore_index=True)
print("\n=== Raw Data Summary ===")
print(f"Shape: {raw_df.shape}")
print(f"Columns: {raw_df.columns.tolist()}")
print("\nDtypes:")
print(raw_df.dtypes)
print("\nTop missing values:")
print(raw_df.isna().sum().sort_values(ascending=False).head(20))

if failed_files:
    print("\nCanh bao: mot so file khong doc duoc:")
    for fp, err in failed_files[:10]:
        print(f"- {fp}: {err}")

raw_df.head()

Tong so file tabular tim thay trong preprocessed: 1

=== Raw Data Summary ===
Shape: (19, 10)
Columns: ['label_source', 'normalization', 'class_weights', 'shapes', 'window_config', 'label_distribution', 'outliers_removed', 'outliers_pct', 'imbalance_ratio', 'source_file']

Dtypes:
label_source           object
normalization          object
class_weights         float64
shapes                 object
window_config         float64
label_distribution     object
outliers_removed        int64
outliers_pct            int64
imbalance_ratio       float64
source_file            object
dtype: object

Top missing values:
normalization         16
label_distribution    16
window_config         16
class_weights         15
shapes                13
label_source           0
outliers_removed       0
outliers_pct           0
imbalance_ratio        0
source_file            0
dtype: int64


,label_source,normalization,class_weights,shapes,window_config,label_distribution,outliers_removed,outliers_pct,imbalance_ratio,source_file
0,Label_from_folder (derived from directory name...,minmax,NaN,NaN,NaN,NaN,0,0,2.166721,preprocessing_stats.json
1,Label_from_folder (derived from directory name...,-0.496983,NaN,NaN,NaN,NaN,0,0,2.166721,preprocessing_stats.json
2,Label_from_folder (derived from directory name...,1.299449,NaN,NaN,NaN,NaN,0,0,2.166721,preprocessing_stats.json
3,Label_from_folder (derived from directory name...,NaN,0.964194,NaN,NaN,NaN,0,0,2.166721,preprocessing_stats.json
4,Label_from_folder (derived from directory name...,NaN,0.778832,NaN,NaN,NaN,0,0,2.166721,preprocessing_stats.json


## 3) Validate Schema + Trich Xuat Feature HRV
Kiem tra cac cot bat buoc (`Time`, `Voltage`, `Peak`, `source_file`), sau do trich xuat feature theo `window_id` hoac `source_file`/window thoi gian.

In [ ]:
REQUIRED_COLS = ["Time", "Voltage", "Peak", "source_file"]
LABEL_CANDIDATES = ["label", "Label", "stress", "stress_level", "target", "class"]
WINDOW_SEC = 30.0
MIN_RR_COUNT = 3


def find_label_column(df_in: pd.DataFrame):
    for c in LABEL_CANDIDATES:
        if c in df_in.columns:
            return c
    return None


def load_feature_candidates() -> pd.DataFrame:
    data_root = PREPROCESSED_DIR.parent
    candidates = [
        FEATURE_FILE,
        data_root / "hrv_features_label.csv",
        data_root / "hrv_features.csv",
    ]

    for fp in candidates:
        if fp.exists() and fp.is_file():
            try:
                tmp = pd.read_csv(fp)
                if not tmp.empty:
                    print(f"Fallback: su dung feature file co san -> {fp}")
                    return tmp.copy()
            except Exception as ex:
                print(f"Khong doc duoc {fp}: {ex}")

    return pd.DataFrame()


def standardize_features_df(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    out = out.loc[:, ~out.columns.astype(str).str.startswith("Unnamed:")].copy()

    label_col = find_label_column(out)
    if label_col is None:
        raise ValueError(
            "Khong tim thay cot label trong feature dataset. Can mot trong cac cot: "
            + ", ".join(LABEL_CANDIDATES)
        )

    if label_col != "label":
        out = out.rename(columns={label_col: "label"})

    out = out.dropna(subset=["label"]).copy()
    if out.empty:
        raise ValueError("Feature dataset rong sau khi loai bo dong thieu label.")

    if "source_file" not in out.columns:
        out["source_file"] = "precomputed_features"
    if "window_id" not in out.columns:
        out["window_id"] = np.arange(len(out))

    if out["label"].dtype == "O" or str(out["label"].dtype).startswith("category"):
        out["label"] = out["label"].astype(str).str.strip()
    else:
        out["label"] = pd.to_numeric(out["label"], errors="coerce")

    out = out.dropna(subset=["label"]).copy()
    if out.empty:
        raise ValueError("Khong con mau nao sau khi chuan hoa cot label.")

    return out


missing_required = [c for c in REQUIRED_COLS if c not in raw_df.columns]

if missing_required:
    print(
        "Canh bao: thieu cot raw waveform (" + ", ".join(missing_required) + "). "
        "Notebook chuyen sang che do train tu feature co san."
    )
    fallback_df = load_feature_candidates()
    if fallback_df.empty:
        fallback_df = raw_df.copy()
    features_df = standardize_features_df(fallback_df)
else:
    df = raw_df.copy()
    for c in ["Time", "Voltage", "Peak"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["Time", "Voltage", "Peak", "source_file"]).copy()

    if df.empty:
        raise ValueError("Du lieu rong sau khi lam sach cot Time/Voltage/Peak/source_file.")

    def infer_label(group_df: pd.DataFrame, source_name: str):
        for col in LABEL_CANDIDATES:
            if col in group_df.columns and group_df[col].notna().any():
                v = group_df[col].dropna().mode().iloc[0]
                try:
                    return int(float(v))
                except Exception:
                    return str(v)

        src = str(source_name).lower()
        patterns = [
            r"(?:pure|mixed)[_/](\d)",
            r"(?:pure|mixed)_(\d)",
            r"[/_](\d)(?:[/_]|$)",
        ]
        for p in patterns:
            m = re.search(p, src)
            if m:
                return int(m.group(1))

        keyword_map = {
            0: ["normal", "relax", "calm", "pure_0"],
            1: ["low", "light", "mild", "pure_1", "mixed_1"],
            2: ["moderate", "medium", "pure_2", "mixed_2"],
            3: ["high", "severe", "stress", "pure_3", "mixed_3"],
        }
        for k, words in keyword_map.items():
            if any(w in src for w in words):
                return k

        return np.nan

    def safe_skew(x: np.ndarray) -> float:
        if len(x) < 3:
            return np.nan
        return float(pd.Series(x).skew())

    def safe_kurt(x: np.ndarray) -> float:
        if len(x) < 4:
            return np.nan
        return float(pd.Series(x).kurt())

    def rr_frequency_features(rr_ms: np.ndarray):
        if len(rr_ms) < 4:
            return np.nan, np.nan, np.nan, np.nan

        rr_s = rr_ms / 1000.0
        t = np.cumsum(rr_s)
        t = t - t[0]
        if t[-1] <= 0:
            return np.nan, np.nan, np.nan, np.nan

        fs = 4.0
        t_uniform = np.arange(0, t[-1], 1 / fs)
        if len(t_uniform) < 8:
            return np.nan, np.nan, np.nan, np.nan

        rr_interp = np.interp(t_uniform, t, rr_ms)
        rr_detrended = rr_interp - np.nanmean(rr_interp)
        fft_vals = np.fft.rfft(rr_detrended)
        freqs = np.fft.rfftfreq(len(rr_detrended), d=1 / fs)
        psd = (np.abs(fft_vals) ** 2) / max(len(rr_detrended), 1)

        lf_mask = (freqs >= 0.04) & (freqs < 0.15)
        hf_mask = (freqs >= 0.15) & (freqs <= 0.40)
        total_mask = (freqs >= 0.04) & (freqs <= 0.40)

        lf_power = float(np.trapz(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else np.nan
        hf_power = float(np.trapz(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else np.nan
        total_power = float(np.trapz(psd[total_mask], freqs[total_mask])) if np.any(total_mask) else np.nan
        lf_hf_ratio = (lf_power / hf_power) if (pd.notna(lf_power) and pd.notna(hf_power) and hf_power > 0) else np.nan

        return lf_power, hf_power, lf_hf_ratio, total_power

    def extract_features_from_group(g: pd.DataFrame, source_name: str, window_id=np.nan):
        g = g.sort_values("Time")
        peak_times = g.loc[g["Peak"] > 0, "Time"].values

        if len(peak_times) < 4:
            return None, "too_few_peaks"

        rr_ms = np.diff(peak_times) * 1000.0
        rr_ms = rr_ms[(rr_ms >= 300.0) & (rr_ms <= 2000.0)]
        if len(rr_ms) < MIN_RR_COUNT:
            return None, "too_few_rr_after_filter"

        rmssd = np.sqrt(np.mean(np.diff(rr_ms) ** 2)) if len(rr_ms) >= 2 else np.nan
        pnn50 = float(np.mean(np.abs(np.diff(rr_ms)) > 50.0) * 100.0) if len(rr_ms) >= 2 else np.nan
        mean_rr_ms = float(np.mean(rr_ms))
        hr_bpm = 60000.0 / mean_rr_ms if mean_rr_ms > 0 else np.nan

        duration_sec = float(g["Time"].max() - g["Time"].min())
        n_peaks = int(np.sum(g["Peak"] > 0))
        peak_rate = n_peaks / duration_sec if duration_sec > 0 else np.nan

        lf_power, hf_power, lf_hf_ratio, total_power = rr_frequency_features(rr_ms)
        label_val = infer_label(g, source_name)

        row = {
            "source_file": source_name,
            "window_id": window_id,
            "label": label_val,
            "mean_rr_ms": mean_rr_ms,
            "median_rr_ms": float(np.median(rr_ms)),
            "std_rr_ms": float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else np.nan,
            "rmssd_ms": float(rmssd),
            "pnn50": float(pnn50),
            "heart_rate_bpm": float(hr_bpm),
            "n_peaks": n_peaks,
            "duration_sec": duration_sec,
            "peak_rate_per_sec": float(peak_rate),
            "voltage_mean": float(g["Voltage"].mean()),
            "voltage_std": float(g["Voltage"].std(ddof=1)) if len(g) > 1 else np.nan,
            "voltage_min": float(g["Voltage"].min()),
            "voltage_max": float(g["Voltage"].max()),
            "voltage_skew": safe_skew(g["Voltage"].values),
            "voltage_kurtosis": safe_kurt(g["Voltage"].values),
            "lf_power": lf_power,
            "hf_power": hf_power,
            "lf_hf_ratio": lf_hf_ratio,
            "total_power": total_power,
        }
        return row, None

    if "window_id" in df.columns:
        base_groups = list(df.groupby(["source_file", "window_id"], sort=False))
        print("Grouping strategy: ('source_file', 'window_id')")
    else:
        base_groups = list(df.groupby(["source_file"], sort=False))
        print("Grouping strategy: 'source_file' (+ optional fixed windows 30s)")

    feature_rows = []
    skip_log = {"too_few_peaks": 0, "too_few_rr_after_filter": 0}

    for key, g in base_groups:
        if isinstance(key, tuple):
            source_name = str(key[0])
            w_id = key[1] if len(key) > 1 else np.nan
        else:
            source_name = str(key)
            w_id = np.nan

        if "window_id" not in df.columns:
            duration = float(g["Time"].max() - g["Time"].min())
            if duration > WINDOW_SEC * 1.5:
                g2 = g.copy()
                g2["_window"] = ((g2["Time"] - g2["Time"].min()) // WINDOW_SEC).astype(int)
                for wv, gw in g2.groupby("_window", sort=False):
                    row, reason = extract_features_from_group(gw, source_name, window_id=int(wv))
                    if row is not None:
                        feature_rows.append(row)
                    else:
                        skip_log[reason] = skip_log.get(reason, 0) + 1
                continue

        row, reason = extract_features_from_group(g, source_name, window_id=w_id)
        if row is not None:
            feature_rows.append(row)
        else:
            skip_log[reason] = skip_log.get(reason, 0) + 1

    features_df = pd.DataFrame(feature_rows)
    if features_df.empty:
        raise ValueError(
            "Khong trich xuat duoc feature nao. Kiem tra cot Peak/Time hoac giam dieu kien loc RR [300, 2000] ms."
        )

    before_drop = len(features_df)
    features_df = features_df.dropna(subset=["label"]).copy()
    print(f"Rows dropped do thieu label: {before_drop - len(features_df)}")

    feature_cols = [c for c in features_df.columns if c not in ["source_file", "window_id", "label"]]
    nan_ratio = features_df[feature_cols].isna().mean(axis=1)
    features_df = features_df.loc[nan_ratio <= 0.6].copy()

    if features_df.empty:
        raise ValueError("Khong con du lieu sau buoc clean features (NaN > 60% bi loai).")

features_df = standardize_features_df(features_df)
features_df.to_csv(FEATURE_FILE, index=False)
print(f"\nSaved features file: {FEATURE_FILE}")
print(f"Feature shape: {features_df.shape}")
print("\nClass distribution (label):")
print(features_df["label"].value_counts(dropna=False).sort_index())
display(features_df.head())

ValueError: Thieu cot bat buoc de trich xuat feature: Time, Voltage, Peak. Hay dam bao du lieu raw sample co day du Time/Voltage/Peak/source_file.

## 4) EDA Tren Feature Dataset
Kiem tra thong ke va phan phoi label tren bo feature da trich xuat.

In [ ]:
if "label" not in features_df.columns:
    raise ValueError("Khong co cot label trong features_df de train classifier.")

print("Feature dataset preview:")
display(features_df.head())

print("\nDescribe numeric features:")
display(features_df.describe(include=[np.number]).T.head(20))

plt.figure(figsize=(8, 4))
sns.countplot(data=features_df, x="label", order=features_df["label"].value_counts().index)
plt.title("Label Distribution In Features")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

num_cols = [c for c in features_df.select_dtypes(include=[np.number]).columns if c != "label"]
if len(num_cols) >= 2:
    plt.figure(figsize=(10, 8))
    sns.heatmap(features_df[num_cols].corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

NameError: name 'target_col' is not defined

## 5) Preprocessing
Dung `ColumnTransformer` + `Pipeline` de impute + scale + one-hot ma khong gay leakage.

In [ ]:
data_model = features_df.copy()

# Cot giu de trace/ID, khong dua vao feature train
leakage_or_trace_cols = [c for c in ["source_file", "window_id"] if c in data_model.columns]

X = data_model.drop(columns=["label"] + leakage_or_trace_cols)
y_raw = data_model["label"]

if y_raw.nunique(dropna=True) < 2:
    raise ValueError("Target label chi co 1 lop, khong the train classifier.")

label_encoder = None
if y_raw.dtype == "O" or str(y_raw.dtype).startswith("category"):
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw.astype(str))
    class_names = label_encoder.classes_.tolist()
    print(f"LabelEncoder applied. Classes: {class_names}")
else:
    y = y_raw.astype(int).values
    class_names = sorted(pd.Series(y).dropna().unique().tolist())

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

if len(numeric_features) + len(categorical_features) == 0:
    raise ValueError("Khong con feature hop le de train sau khi loai leakage columns.")

print(f"Modeling samples: {len(X)}")
print(f"Target classes: {class_names}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

feature_columns = X.columns.tolist()
print(f"Feature matrix shape: {X.shape}")

NameError: name 'X' is not defined

## 6) Train/Validation Split
Split theo yeu cau: `test_size=0.2`, `random_state=42`, uu tien `stratify=y` neu hop le.

In [ ]:
y_series = pd.Series(y)
n_samples = len(X)

if n_samples < 2:
    raise ValueError(
        "So mau sau buoc feature extraction qua it (<2), khong the split train/test."
    )

if y_series.nunique() < 2:
    raise ValueError(
        "Target chi co 1 lop trong du lieu modeling, khong the train classifier."
    )

class_counts = y_series.value_counts()
can_stratify = class_counts.min() >= 2 and y_series.nunique() > 1 and n_samples >= 4

stratify_arg = y if can_stratify else None
if not can_stratify:
    print("Canh bao: khong the stratify do co lop qua it mau. Se split khong stratify.")

if n_samples == 2:
    test_size = 0.5
elif n_samples == 3:
    test_size = 1 / 3
else:
    test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=RANDOM_STATE,
    stratify=stratify_arg,
)

print(f"test_size su dung: {test_size}")
print(f"X_train: {X_train.shape}, y_train: {len(y_train)}")
print(f"X_test:  {X_test.shape}, y_test:  {len(y_test)}")
print("Class distribution in train:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

NameError: name 'pd' is not defined

## 7) Huan Luyen Nhieu Mo Hinh
Train toi thieu 4 mo hinh: LogisticRegression, RandomForest, SVC, GradientBoosting.
Bo sung KNN de so sanh them.

In [ ]:
def make_pipeline(model):
    return Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])


is_binary = len(np.unique(y_train)) == 2
primary_scoring = "f1" if is_binary else "f1_weighted"

models = {
    "LogisticRegression": make_pipeline(
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    ),
    "RandomForest": make_pipeline(
        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    ),
    "SVC": make_pipeline(SVC(probability=True, random_state=RANDOM_STATE)),
    "GradientBoosting": make_pipeline(
        GradientBoostingClassifier(random_state=RANDOM_STATE)
    ),
    "KNN": make_pipeline(KNeighborsClassifier(n_neighbors=7)),
}

# Tu chon: XGBoost neu co san
try:
    from xgboost import XGBClassifier

    models["XGBoost"] = make_pipeline(
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            eval_metric="mlogloss",
        )
    )
    print("XGBoost available -> added to candidate models.")
except Exception:
    print("XGBoost khong kha dung trong moi truong hien tai, bo qua.")

print("Models to train:", list(models.keys()))

## 8) Danh Gia Mo Hinh
Danh gia bang: Accuracy, Precision, Recall, F1, Confusion Matrix, Classification Report.
Neu binary, tinh them ROC-AUC va ROC curve.

In [ ]:
def evaluate_model(model_name: str, fitted_model, X_eval, y_eval, is_binary_task: bool):
    y_pred = fitted_model.predict(X_eval)

    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision_weighted": precision_score(y_eval, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_eval, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_eval, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_eval, y_pred, average="macro", zero_division=0),
    }

    if is_binary_task and hasattr(fitted_model, "predict_proba"):
        y_proba = fitted_model.predict_proba(X_eval)[:, 1]
        result["roc_auc"] = roc_auc_score(y_eval, y_proba)
    else:
        result["roc_auc"] = np.nan

    print(f"\n===== {model_name} =====")
    print("Confusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_eval, y_pred, zero_division=0))

    return result


results = []
fitted_models = {}

for model_name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted_models[model_name] = pipe
    metrics = evaluate_model(model_name, pipe, X_test, y_test, is_binary)
    results.append(metrics)

if is_binary:
    plt.figure(figsize=(7, 5))
    for model_name, fitted_model in fitted_models.items():
        if hasattr(fitted_model, "predict_proba"):
            y_proba = fitted_model.predict_proba(X_test)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            plt.plot(fpr, tpr, label=model_name)

    plt.plot([0, 1], [0, 1], "k--", alpha=0.7)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (Binary Classification)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9) Hyperparameter Tuning + So Sanh Mo Hinh
Tuning it nhat 2 mo hinh bang GridSearchCV (uu tien cv=5, tu dong giam neu du lieu qua it).

In [ ]:
tuned_models = {}

logreg_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__class_weight": [None, "balanced"],
}

rf_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
}

min_class_train = int(pd.Series(y_train).value_counts().min())
cv_folds = min(5, min_class_train) if min_class_train >= 2 else 0
if cv_folds < 2:
    print("Canh bao: du lieu train qua it cho cross-validation, bo qua tuning.")
else:
    grid_jobs = [
        ("LogisticRegression_tuned", models["LogisticRegression"], logreg_grid),
        ("RandomForest_tuned", models["RandomForest"], rf_grid),
    ]

    for tuned_name, base_pipe, param_grid in grid_jobs:
        print(f"\nRunning GridSearchCV for: {tuned_name} (cv={cv_folds})")
        gs = GridSearchCV(
            estimator=base_pipe,
            param_grid=param_grid,
            cv=cv_folds,
            scoring=primary_scoring,
            n_jobs=-1,
            verbose=1,
        )
        gs.fit(X_train, y_train)
        tuned_models[tuned_name] = gs.best_estimator_
        print(f"Best params for {tuned_name}: {gs.best_params_}")

        tuned_metrics = evaluate_model(tuned_name, gs.best_estimator_, X_test, y_test, is_binary)
        results.append(tuned_metrics)

    fitted_models.update(tuned_models)

results_df = pd.DataFrame(results)
if results_df.empty:
    raise ValueError("Khong co ket qua model nao de so sanh.")

sort_metric = "f1_weighted" if not is_binary else "f1_weighted"
results_df = results_df.sort_values(by=sort_metric, ascending=False).reset_index(drop=True)

best_model_name = results_df.loc[0, "model"]
best_score = results_df.loc[0, sort_metric]
best_model = fitted_models[best_model_name]

print("\n=== Model Comparison ===")
display(results_df)
print(f"Best model: {best_model_name} ({sort_metric}={best_score:.4f})")

results_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
print(f"Saved comparison report: {(REPORTS_DIR / 'model_comparison.csv').resolve()}")

## 10) Luu Artifacts
Luu best model, danh sach cot feature va label encoder (neu can) de infer lai dung schema.

In [ ]:
best_model_path = ARTIFACTS_DIR / "best_model.joblib"
feature_cols_path = ARTIFACTS_DIR / "feature_columns.joblib"
label_encoder_path = ARTIFACTS_DIR / "label_encoder.joblib"

joblib.dump(best_model, best_model_path)
joblib.dump(feature_columns, feature_cols_path)

if label_encoder is not None:
    joblib.dump(label_encoder, label_encoder_path)
    print(f"Saved label encoder: {label_encoder_path.resolve()}")
else:
    print("Khong can luu label encoder vi label da la so.")

print(f"Saved best model: {best_model_path.resolve()}")
print(f"Saved feature columns: {feature_cols_path.resolve()}")

## 11) Inference Mau Tren Vai Dong Du Lieu
Load model da luu va thu predict tren mot vai dong trong tap test.

In [ ]:
loaded_model = joblib.load(best_model_path)

sample_n = min(5, len(X_test))
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), size=sample_n, replace=False)
X_sample = X_test.iloc[sample_idx].copy()
y_true_sample = np.array(y_test)[sample_idx]
y_pred_sample = loaded_model.predict(X_sample)

infer_df = pd.DataFrame({
    "y_true": y_true_sample,
    "y_pred": y_pred_sample,
})

if hasattr(loaded_model, "predict_proba"):
    y_proba = loaded_model.predict_proba(X_sample)
    infer_df["pred_confidence"] = y_proba.max(axis=1)

if label_encoder is not None:
    infer_df["y_true_label"] = label_encoder.inverse_transform(infer_df["y_true"].astype(int))
    infer_df["y_pred_label"] = label_encoder.inverse_transform(infer_df["y_pred"].astype(int))

display(infer_df)

## 12) Ket Luan Va Huong Phat Trien
- Da tao pipeline tu raw sample (`Time`, `Voltage`, `Peak`, `source_file`) -> feature HRV.
- Da luu feature dataset tai `data/features/features_hrv.csv`.
- Da train >= 4 model, co tuning >= 2 model (neu du mau cho CV).
- Da luu model tot nhat tai `artifacts/best_model.joblib` va bang so sanh tai `reports/model_comparison.csv`.

Huong phat trien tiep theo:
1. Bo sung quality index cho signal va detector artifact nang cao hon cho RR.
2. Them nested CV/time-aware split neu co tinh chat chuoi thoi gian.
3. Thu LightGBM/XGBoost va calibration de cai thien kha nang tong quat hoa.